# $B^+\to K^+\pi^+\pi^-$ fit with efficiency and background

Non-CP fit with efficiency, background, displaced start values, and fit-fraction closure. Dalitz variables: $s_{13}=m^2(K^+\pi^-)$ and $s_{23}=m^2(\pi^+\pi^-)$.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from dalitzplotfitter import BaBarFlatte, DecayChannel, DecayModel, LASS, Minimizer, NonResonant, Parameter, PhaseSpaceSample, RealImag, Resonance, enable_x64, weighted_resample
from dalitzplotfitter.background import FunctionalBackground
from dalitzplotfitter.efficiency import FunctionalEfficiency
enable_x64()


## 1. Amplitude model


In [ ]:
channel=DecayChannel("B+",("K+","pi+","pi-"))
truth_xy={"Kstar892":(1.0,0.0),"KpiS":(1.40,-0.60),"rho770":(0.65,0.10),"f0_980":(-0.20,1.00),"NR":(-0.50,0.10)}
truth={}
def coefficient(name,fixed=False):
    x,y=truth_xy[name]
    if fixed:return RealImag(x,y)
    truth[f"{name}.x"],truth[f"{name}.y"]=x,y
    return RealImag(Parameter.coefficient(f"{name}.x",x,owner=name,step=0.01),Parameter.coefficient(f"{name}.y",y,owner=name,step=0.01))
c={n:coefficient(n,fixed=(n=="Kstar892")) for n in truth_xy}
components=[
 Resonance("Kstar892",(0,2),c["Kstar892"],mass=0.8958,width=0.0474,spin=1,resonance_radius=4.0,parent_radius=4.0),
 Resonance("KpiS",(0,2),c["KpiS"],lineshape=LASS(2.07,3.32,1.8),mass=1.425,width=0.270,spin=0,resonance_radius=4.0,parent_radius=4.0),
 Resonance("rho770",(1,2),c["rho770"],mass=0.7753,width=0.1491,spin=1,resonance_radius=4.0,parent_radius=4.0),
 Resonance("f0_980",(1,2),c["f0_980"],lineshape=BaBarFlatte(),mass=0.965,width=0.0,spin=0,resonance_radius=4.0,parent_radius=4.0),
 NonResonant(c["NR"])]
model=DecayModel(channel,components,normalization_method="square-dalitz",normalization_resolution=350,normalization_pair=(0,2))
norm=model.normalization_sample
print("Generated fit fractions:"); model.print_fit_fractions(truth,normalization_sample=norm,include_interference=True)


## 2. Efficiency and background


In [ ]:
s13_min=(channel.daughter_masses[0]+channel.daughter_masses[2])**2; s13_max=(channel.parent_mass-channel.daughter_masses[1])**2
s23_min=(channel.daughter_masses[1]+channel.daughter_masses[2])**2; s23_max=(channel.parent_mass-channel.daughter_masses[0])**2
def scaled(d,k,lo,hi):return jnp.clip((d[k]-lo)/(hi-lo),0.0,1.0)
efficiency=FunctionalEfficiency(lambda d:0.55+0.30*scaled(d,"s13",s13_min,s13_max)+0.10*jnp.cos(jnp.pi*scaled(d,"s23",s23_min,s23_max)))
background=FunctionalBackground(lambda d:0.50+1.20*scaled(d,"s13",s13_min,s13_max)+0.40*scaled(d,"s23",s23_min,s23_max))
eff_norm=efficiency(norm.as_dict()); bkg_norm=jnp.mean(norm.weights*background(norm.as_dict()))


## 3. Generate pseudo-data


In [ ]:
N_POOL=250_000; N_DATA=40_000; BACKGROUND_FRACTION_TRUE=0.18
pool=model.generate_phase_space(N_POOL,seed=2028); pool_cache=model.prepare_cache(pool,norm)
signal_weights=pool.weights*efficiency(pool.as_dict())*pool_cache.intensity(truth); background_weights=pool.weights*background(pool.as_dict())
n_background=int(round(N_DATA*BACKGROUND_FRACTION_TRUE)); n_signal=N_DATA-n_background
signal_data=weighted_resample(jax.random.key(2029),pool,signal_weights,n_signal,replace=True); background_data=weighted_resample(jax.random.key(2030),pool,background_weights,n_background,replace=True)
def merge(a,b):
    def joined(name):
        x,y=getattr(a,name),getattr(b,name); return None if x is None else jnp.concatenate((x,y))
    return PhaseSpaceSample(s12=joined("s12"),s13=joined("s13"),s23=joined("s23"),weights=jnp.ones((a.size+b.size,)),p1=joined("p1"),p2=joined("p2"),p3=joined("p3"))
data=merge(signal_data,background_data)
fig,ax=plt.subplots(figsize=(7,5.5)); h=ax.hist2d(np.asarray(data.s13),np.asarray(data.s23),bins=90); fig.colorbar(h[3],ax=ax,label="events"); ax.set(xlabel=r"$s_{13}$ [GeV$^2$]",ylabel=r"$s_{23}$ [GeV$^2$]"); plt.show()


## 4. Fit


In [ ]:
cache=model.prepare_cache(data,norm,efficiency_normalization=eff_norm); eff_data=efficiency(data.as_dict()); bkg_data=background(data.as_dict())/bkg_norm
background_fraction=Parameter("background_fraction",0.12,bounds=(0.001,0.50),step=0.01); fit_parameters=(*model.parameters,background_fraction)
def nll(values):
    signal_pdf=eff_data*cache.intensity(values)/cache.normalization(values); f=values["background_fraction"]; return -jnp.sum(jnp.log(jnp.clip((1-f)*signal_pdf+f*bkg_data,min=1e-300)))
rng=np.random.default_rng(314159)
start={p.name:truth[p.name]+rng.normal(0.0,0.12) for p in model.parameters if not p.fixed}
start["background_fraction"]=float(np.clip(BACKGROUND_FRACTION_TRUE+rng.normal(0.0,0.04),0.01,0.49))
result=Minimizer(nll,fit_parameters,verbose=1).fit(start_values=start,simplex=True,ncall=40_000)
fit_values={p.name:float(result.values[p.name]) for p in model.parameters if not p.fixed}
print("valid:",result.valid,"NLL:",result.fval,"EDM:",result.fmin.edm)
print(f"{'parameter':20s} {'generated':>11s} {'start':>11s} {'fitted':>11s} {'error':>11s} {'pull':>9s}")
for p in model.parameters:
    if p.fixed:continue
    fit=float(result.values[p.name]); err=float(result.errors[p.name]); print(f"{p.name:20s} {truth[p.name]:11.5f} {start[p.name]:11.5f} {fit:11.5f} {err:11.5f} {(fit-truth[p.name])/err:9.3f}")
bf=float(result.values["background_fraction"]); be=float(result.errors["background_fraction"]); print(f"{'background_fraction':20s} {BACKGROUND_FRACTION_TRUE:11.5f} {start['background_fraction']:11.5f} {bf:11.5f} {be:11.5f} {(bf-BACKGROUND_FRACTION_TRUE)/be:9.3f}")
print("\nFitted fit fractions (without efficiency):"); model.print_fit_fractions(fit_values,normalization_sample=norm,include_interference=True)
print("\nFitted fit fractions (with efficiency):"); model.print_fit_fractions(fit_values,normalization_sample=norm,efficiency=efficiency,include_interference=True)


## 5. Projections


In [ ]:
projection_cache=model.prepare_cache(pool,norm,efficiency_normalization=eff_norm); efficiency_pool=efficiency(pool.as_dict()); background_pool=background(pool.as_dict())/bkg_norm
def proj(values,f,var,bins):
    signal=pool.weights*efficiency_pool*projection_cache.intensity(values)/projection_cache.normalization(values); mix=(1-f)*signal+f*pool.weights*background_pool
    return np.histogram(np.asarray(getattr(pool,var)),bins=bins,weights=np.asarray(mix))[0]
fig,axes=plt.subplots(1,2,figsize=(13,4.8),constrained_layout=True)
for ax,var,label in zip(axes,("s13","s23"),(r"$s_{13}$ [GeV$^2$]",r"$s_{23}$ [GeV$^2$]")):
    obs=np.asarray(getattr(data,var)); bins=np.linspace(obs.min(),obs.max(),70); centers=0.5*(bins[:-1]+bins[1:]); dh=np.histogram(obs,bins=bins)[0]
    gh=proj(truth,BACKGROUND_FRACTION_TRUE,var,bins); fh=proj(fit_values,bf,var,bins); gh*=dh.sum()/gh.sum(); fh*=dh.sum()/fh.sum()
    ax.errorbar(centers,dh,yerr=np.sqrt(np.maximum(dh,1)),fmt=".",label="toy data"); ax.step(centers,gh,where="mid",linestyle="--",label="generated model"); ax.step(centers,fh,where="mid",label="fitted model"); ax.set(xlabel=label,ylabel="events / bin"); ax.legend()
plt.show()
